# NB06 — Stage B technique sweep (run **after NB07**)

This notebook has no fallback architecture list. It downloads the locked XAI
selection from the public HF dataset and stops if that file is absent. That
gate prevents an arbitrary accuracy ranking from launching hundreds of runs.
The completed public `2026-08-30-r3` gate selected `regnety016`,
`densenet121`, and `resnet50`; cell 2 still re-downloads and verifies the
selection and its raw evidence so the notebook never relies on this prose.

**Memory-repair revision v5 (2026-08-31):** public telemetry proved the kernel
deaths were host-RAM exhaustion, not GPU OOM. The earlier ROI-only diagnosis was
incomplete: full-frame arms also accumulated process RSS. Large temporary
checkpoint/upload arenas were being left mapped in the long-lived Python
process. v5 serialises one full checkpoint per epoch (the best file is an atomic
snapshot, not a second serialisation), releases free Linux arenas after saves,
loads and HF commits, and stops the whole worker after a clean 88% RAM pause.
The ROI crop/loader repairs remain. The locked architectures, batch size,
resolution, optimiser, and 60-epoch scientific recipe are unchanged.

**Epoch-history repair v6 (2026-09-01):** v5 added a commit-policy telemetry
field while two 60-epoch runs still had v4 CSV headers. Positional append wrote
178 values under 177 headings, so pandas stopped resume with `ParserError` even
though both checkpoints were intact. v6 recognises that exact revision marker,
inserts a blank for the older rows, verifies every row width, and atomically
rewrites the canonical table. Future epoch writes merge by column name and can
expand the header safely. Unknown drift raises without modifying the source
file; no epoch or metric is silently dropped.

**Single-notebook/progress repair v10 (2026-09-02):** the submitted output did
not run in the requested one-notebook mode: its own banner says
`worker=0/4`, and the planner therefore reserved three quarters of the sweep
for other accounts. The session cell now has one unambiguous source of truth,
`ACTIVE_KAGGLE_ACCOUNTS`, and defaults to `('acct1',)`. `NUM_WORKERS` and
`WORKER_ID` are derived and cannot silently disagree. Training now prints a
plain-text heartbeat as soon as batch 1 finishes in every epoch, because a
saved Kaggle `tqdm` widget may remain at `0%` even while the kernel is working.
The attached output already completed epoch 37 in 3.7 minutes, and the public
HF checkpoint subsequently completed all 60 epochs; no model or scientific
setting changed. The PyTorch weight-norm telemetry warning is also removed by
an explicit no-gradient detach. The `[LOADER] workers=0` message now states
that it counts CPU input helpers—not training/GPU workers—so the memory-safe
synchronous loader is not mistaken for an idle model.

**Process-isolation repair v11 (2026-09-03):** v10 completed two models and
then trained a third from epoch 3 through epoch 45, but the long-lived Jupyter
process retained memory after every epoch. Public telemetry measures a steady
RSS rise of 0.17 GB/epoch for both RegNet runs (and 0.30 GB/epoch on the
inspected DenseNet run), even with synchronous loading, buffer draining,
`gc.collect`, and `malloc_trim`. At 88.1% the safety guard correctly published
epoch 45 and stopped the cell. v11 runs each model in a disposable child Python
process. When that run finishes—or pauses—Linux destroys the whole child and
reclaims its model, optimiser, CUDA, serialization, image-library, and allocator
state. If a child reaches the RAM guard, the parent immediately starts a clean
child that resumes the **same** HF checkpoint, instead of ending the notebook.
The parent still owns scheduling, the 45-minute end-of-session guard, and the
final repository reconciliation. No experimental setting changed.

**Account/progress-label repair v12 (2026-09-08):** Python interprets
`('acct1')` as text rather than a one-item tuple. The attached run stopped in
cell 3 for exactly that reason, before data loading or training. The shared
session cell now normalises either form to `('acct1',)` and still validates
four-account tuples. Final HF verification also separates genuinely unsafe
partial artifacts (`AT RISK`) from runs with no files (`NOT STARTED`). No model,
checkpoint, or training recipe changed.

**CUDA/scheduler/commit repair revision (2026-08-31):** two independent public
RegNetY-16GF ROI attempts failed on their first batch in the same cuDNN grouped
convolution with `CUDNN_STATUS_EXECUTION_FAILED` / `misaligned address`, while
each T4 held only about 1.1 GB. RegNet therefore keeps the exact same model,
384px input, batch 32, AMP, and optimiser but uses the conservative contiguous
(NCHW) cuDNN path instead of `channels_last`; all other architectures keep the
Stage-A layout. A fatal CUDA context now stops the session only after publishing
the error, rather than cascading into unrelated runs. Fresh work is also kept
with its static owner: an absent run is no longer mislabelled as a dead worker
and stolen by all four accounts at once. Work stealing is now opt-in and is
disabled here. An ordinary static-owner claim is batched into the 30-minute HF
cycle; it no longer burns one immediate commit before every model. A paused run
ends the training cell instead of cascading through dozens of one-epoch runs.

Before planning hours of work, cell 2 runs one exact dual-T4 RegNet
forward/backward/optimizer step in an **isolated child process** and publishes
the log. If the Kaggle CUDA image still rejects the conservative profile, only
the child process is poisoned and NB06 stops before claiming a training run.

**Stop every older v4-v11 NB06 copy before starting v12.** For one Kaggle copy,
leave `ACTIVE_KAGGLE_ACCOUNTS=('acct1',)` and `ACCOUNT='acct1'`. For four
parallel copies, put all four labels in `ACTIVE_KAGGLE_ACCOUNTS` in every copy
and set `ACCOUNT` to that copy's label. `NUM_WORKERS` and `WORKER_ID` are
derived automatically. Cell 4 reads the current
public HF state: completed runs are skipped and every partial run resumes from
its published `ckpt_last.pt`. No architecture or completed epoch is discarded.

Every arm below changes one declared factor relative to the Stage-A recipe.
The tyre-ROI arm runs first. Unsupported values fail before training rather
than becoming silent no-op experiments. Until the tyre groups are re-cut, the
sweep runs **fold 1 only** (three seeds): folds 0 and 2 carry the known
cross-fold-tyre warning and are already saturated, so spending two-thirds of
the Stage-B budget there cannot measure a technique effect.


In [1]:
# === CELL 1 of every notebook: unpack the library ==========================
# Writes tyrelib.py into the session and imports it. Nothing here touches the
# GPU or the network beyond installing three small packages.
#
#   tyrelib   the whole pipeline: HuggingFace sync, registry, work sharding,
#             telemetry, model zoo, training loop, metrics.
#
# Generated by build_notebooks.py from tyrelib.py. Editing the blob below does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle ships torch, pandas, sklearn. These vary by image version, so check.
#   pynvml  reads GPU power/temperature/clocks directly (per device)
#   psutil  peak RAM and CPU
#   pyarrow writes per-sample predictions as Parquet
for _pkg in ('pynvml', 'psutil', 'pyarrow', 'timm'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg],
                       check=False)

_LIB = (
    '...',
    'MCBpZiBzZWxmdGVzdCgpIGVsc2UgMSkK',
)

(WORK / 'tyrelib.py').write_bytes(base64.b64decode(''.join(_LIB)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
# Without this, re-running cell 1 after an edit returns the cached module and
# you spend an hour debugging a ghost.
for _m in [m for m in list(sys.modules) if m == 'tyrelib']:
    del sys.modules[_m]
import tyrelib as tl
print('tyrelib', tl.__version__, 'loaded')


tyrelib v12 loaded


## 1 — Session and data

In [2]:
# === Who am I? =============================================================
#
# ACCOUNT labels this Kaggle account in the shared run log.
# ACTIVE_KAGGLE_ACCOUNTS is the ONE source of truth for parallelism.
# NUM_WORKERS and WORKER_ID are derived from it; do not edit them.
#
# ---------------------------------------------------------------------------
# THESE TWO VALUES ARE SAFE TO CHANGE AT ANY TIME.
#
# They decide which account owns each fresh run. An absent or partial run stays
# with that static owner. Work stealing is disabled by default; an explicit
# recovery run may opt in and can then take over only a real claim/run event
# older than 45 minutes. Whether a run is finished, and what epoch it reached,
# is read from HuggingFace -- from the run's own files -- so it is the same
# answer for every account at every worker count.
# Go from 4 workers to 1 and nothing is retrained: the runs the other three
# finished are skipped, and the ones they left half-done are RESUMED from
# their checkpoints.
#
# (It did not always work that way. Resume used to check only the local disk,
#  and Kaggle wipes that between sessions, so every run restarted at epoch 1.
#  See docs/05 -- Bug 8.)
# ---------------------------------------------------------------------------
#
# All accounts push to the SAME HuggingFace account (Shanmuk4622), so the
# 128-writes-per-hour budget is SHARED. tyrelib caps each worker at
# 100/NUM_WORKERS automatically.
#
# DEFAULT: exactly one Kaggle notebook. For four parallel copies, replace the
# tuple with ('acct1', 'acct2', 'acct3', 'acct4') in every copy and set ACCOUNT
# to that copy's label. Keeping the active labels in one tuple prevents a cell
# that says "one worker" in one place but silently launches as worker 0/4.
ACTIVE_KAGGLE_ACCOUNTS = ('acct1', 'acct2', 'acct3', 'acct4')   # <<< one notebook; list all four only when all four run
ACCOUNT = 'acct1'                     # <<< this copy's label

# A missing comma in ('acct1') makes it a string. tyrelib deliberately repairs
# that common edit, so a valid one-worker session cannot fail before HF sync.
ACTIVE_KAGGLE_ACCOUNTS = tl.normalise_active_accounts(ACTIVE_KAGGLE_ACCOUNTS)
if ACCOUNT not in ACTIVE_KAGGLE_ACCOUNTS:
    raise ValueError(f"ACCOUNT={ACCOUNT!r} is not active: {ACTIVE_KAGGLE_ACCOUNTS}")
NUM_WORKERS = len(ACTIVE_KAGGLE_ACCOUNTS)
WORKER_ID = ACTIVE_KAGGLE_ACCOUNTS.index(ACCOUNT)
print(f"RUN MODE CHECK: {ACCOUNT=} {ACTIVE_KAGGLE_ACCOUNTS=} -> worker {WORKER_ID}/{NUM_WORKERS}")

sess = tl.Session(account=ACCOUNT, worker_id=WORKER_ID, num_workers=NUM_WORKERS,
                  stage='b',
                  hf_repo='Shanmuk4622/tyre-wear-study',
                  enable_hf=True,
                  session_limit_h=8.5,      # push + pause before Kaggle kills us
                  push_interval_min=30)     # background commit cycle


RUN MODE CHECK: ACCOUNT='acct1' ACTIVE_KAGGLE_ACCOUNTS=('acct1', 'acct2', 'acct3', 'acct4') -> worker 0/4
[DISK] staging /kaggle/temp/tyre_study  (1102 GB free)
[HF] authenticated as Shanmuk4622  ->  dataset:Shanmuk4622/tyre-wear-study
[HF] rate cap 25/hr (shared across all workers on this token)
[HF] background uploader started (30 min cycle)
[LIFE] guards installed (SIGTERM, SIGINT, atexit, watchdog @ 8.5 h)

[SESSION] account=acct1  worker=0/4  stage=b  id=e5f741
[SESSION] MODE=4 PARALLEL NOTEBOOKS: each account starts with one static shard, then safely helps when idle
[SESSION] staging /kaggle/temp/tyre_study  |  hf ON  |  cap 25/hr  |  push every 30 min
[SESSION] NUM_WORKERS assigns each FRESH run to one static owner. Completed/resumable state still comes from HuggingFace.



In [3]:
# === Find the dataset ======================================================
# One Kaggle dataset holds the whole package:
#     <slug>/FINAL/{images,splits,manifests}
#     <slug>/annotations/{clean,propagated}
# Kaggle sometimes wraps uploads in one more directory, so both are searched for.
DATA_ROOT = sess.prepare_data()
ANN_ROOT  = tl.find_annotations_root(DATA_ROOT)
print("annotations:", ANN_ROOT if ANN_ROOT else "NOT FOUND (only needed from NB08 onward)")


[DATA] root /kaggle/input/datasets/shanmuk4622/tire-dataset-prepared/Tire Dataset Prepared/FINAL
[DATA] 418 clean / 4180 derivatives / 12 sessions
annotations: /kaggle/input/datasets/shanmuk4622/tire-dataset-prepared/Tire Dataset Prepared/annotations


## 2 — Load the locked Stage-B selection

In [4]:
import pandas as pd
from pathlib import Path
from huggingface_hub import hf_hub_download

try:
    p = hf_hub_download(tl.HF_REPO_DEFAULT, "tables/stage_b_selection.csv",
                        repo_type="dataset", token=None,
                        local_dir=str(Path(sess.stage_dir) / "public_pull"))
except Exception as e:
    raise RuntimeError("NB07 has not published tables/stage_b_selection.csv. "
                       "Run the corrected NB07 first; NB06 is intentionally blocked.") from e
SEL = pd.read_csv(p)
assert ("selection_revision" in SEL and
        SEL.selection_revision.eq("2026-08-30-r3").all()), \
       "stale Stage-B selection; rerun the corrected NB07"
flag = (SEL.selected_top3 if SEL.selected_top3.dtype == bool else
        SEL.selected_top3.astype(str).str.lower().eq("true"))
TOP3 = list(SEL.loc[flag, "arch"])
assert len(TOP3) == 3, f"expected exactly three locked architectures, got {TOP3}"
elig = SEL.loc[flag, "eligible"]
elig = (elig if elig.dtype == bool else elig.astype(str).str.lower().eq("true"))
assert elig.all() and \
       SEL.loc[flag, "seeds"].astype(int).eq(3).all(), \
       "selection is not three-seed confirmed; rerun NB07"
assert SEL.loc[flag, "xai_status"].eq("ok").all(), \
       "a selected architecture is not XAI-valid"

# Independently verify the gate from the raw public evidence rather than
# trusting only the summary booleans. This also reports the true denominator;
# an older NB07 table counted only valid rows in both n_images and n_valid.
ep = hf_hub_download(tl.HF_REPO_DEFAULT, "tables/xai_evidence_all.csv",
                     repo_type="dataset", token=None,
                     local_dir=str(Path(sess.stage_dir) / "public_pull"))
EV = pd.read_csv(ep)
EV["ter_norm"] = pd.to_numeric(EV.ter_norm, errors="coerce")
raw_selected = EV[EV.arch.isin(TOP3) & EV.xai_status.eq("ok")].copy()
raw_coverage = raw_selected.groupby("arch").agg(
    seeds=("seed", "nunique"), n_images=("image_id", "size"),
    n_valid=("ter_norm", "count"))
raw_coverage["coverage"] = raw_coverage.n_valid / raw_coverage.n_images.clip(lower=1)
assert set(raw_coverage.index) == set(TOP3), "selected evidence rows are missing"
assert raw_coverage.seeds.astype(int).eq(3).all(), \
       "selected raw evidence does not contain all three seeds"
assert raw_coverage.n_valid.gt(0).all(), "selected architecture has no valid TER_norm evidence"
print(SEL.round(4).to_string(index=False))
print("\nlocked architectures:", TOP3)
print("\nraw selected-evidence coverage:\n", raw_coverage.round(4).to_string())
tl.assert_zoo_ok(TOP3)
print("\nKaggle CUDA runtime layouts (model/config unchanged):")
for arch in TOP3:
    print(f"  {arch:14s} {tl.training_memory_format(arch)}")

# Fatal CUDA launch errors poison a process, so prove the repaired RegNet
# profile in a disposable child before any run is claimed. This is an exact
# dual-GPU training step at the registered batch/resolution, not a one-image
# CPU construction check.
import subprocess, sys
from pathlib import Path
CUDA_PROBE = r"""
import sys, torch, tyrelib as tl
arch, res, bs = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])
dev = torch.device("cuda")
assert torch.cuda.device_count() >= 2, "NB06 requires Kaggle dual T4"
fmt_name = tl.training_memory_format(arch)
fmt = torch.contiguous_format if fmt_name == "contiguous" else torch.channels_last
torch.backends.cudnn.benchmark = fmt_name == "channels_last"
torch.manual_seed(20260831); torch.cuda.manual_seed_all(20260831)
model = tl.build_model(arch, 3, pretrained=False, head="coral",
                       img_size=res).to(dev).to(memory_format=fmt)
model = torch.nn.DataParallel(model).train()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=.05)
scaler = tl._grad_scaler(dev)
x = torch.randn(bs, 3, res, res).to(dev, non_blocking=True).to(memory_format=fmt)
y = torch.arange(bs, device=dev) % 3
opt.zero_grad(set_to_none=True)
with tl._autocast(dev):
    loss = tl.CoralHead.loss(model(x), y)
scaler.scale(loss).backward(); scaler.unscale_(opt)
torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
scaler.step(opt); scaler.update(); torch.cuda.synchronize()
print(f"CUDA_SMOKE_PASS arch={arch} res={res} batch={bs} devices=2 "
      f"layout={fmt_name} cudnn_benchmark={torch.backends.cudnn.benchmark} "
      f"loss={float(loss.detach()):.6f} safety={tl.CUDA_SAFETY_REVISION}")
"""
for arch in sorted(set(TOP3) & set(tl.CUDA_CONTIGUOUS_ARCHS)):
    spec = tl.ZOO[arch]
    result = subprocess.run(
        [sys.executable, "-c", CUDA_PROBE, arch, str(spec["res"]), str(spec["bs"])],
        cwd=str(Path.cwd()), text=True, capture_output=True, timeout=600)
    probe_log = (result.stdout + "\n" + result.stderr).strip() + "\n"
    probe_path = Path(sess.stage_dir) / "preflight" / f"cuda_{arch}_{ACCOUNT}.txt"
    probe_path.parent.mkdir(parents=True, exist_ok=True)
    probe_path.write_text(probe_log)
    sess.uploader.enqueue(probe_path,
                          f"preflight/cuda_profiles/{probe_path.name}", force=True)
    sess.push_now(f"CUDA profile preflight {arch} {ACCOUNT}")
    print(probe_log)
    if result.returncode:
        raise RuntimeError(
            f"isolated CUDA training smoke failed for {arch}; log published to "
            f"preflight/cuda_profiles/{probe_path.name}. No run was claimed."
        )


stage_b_selection.csv: 0.00B [00:00, ?B/s]

xai_evidence_all.csv: 0.00B [00:00, ?B/s]

        arch  seeds  n_images  n_valid  ter_norm  ter_sd    bar  sar  dmgar                        xai_status  xai_coverage  eligible  selected_top3 selection_revision                                                                   selection_rule
  regnety016      3       180      180    1.5785  0.1828 0.0310  0.0 0.0014                                ok           1.0      True           True      2026-08-30-r3 top TER_norm among five seed-confirmed screens; BAR tie-break; accuracy excluded
 densenet121      3       178      178    1.5513  0.1951 0.0455  0.0 0.0006                                ok           1.0      True           True      2026-08-30-r3 top TER_norm among five seed-confirmed screens; BAR tie-break; accuracy excluded
    resnet50      3       180      180    1.5146  0.3232 0.0512  0.0 0.0019                                ok           1.0      True           True      2026-08-30-r3 top TER_norm among five seed-confirmed screens; BAR tie-break; accuracy excluded
conv

No files have been modified since last commit. Skipping to prevent empty commit.


[HF] commit #1: 1 file(s), 0.0 MB, 0.3s  [1/25 this hr]
CUDA_SMOKE_PASS arch=regnety016 res=384 batch=32 devices=2 layout=contiguous cudnn_benchmark=False loss=0.693925 safety=2026-08-31-r1



## 3 — Verify masks once for the ROI control

In [5]:
assert ANN_ROOT is not None, "annotations are required for roi_tyre"
ANN = tl.ensure_annotations(DATA_ROOT, ann_root=ANN_ROOT,
                            work_dir=sess.stage_dir / "annotations")
MASK_ROOTS = {"clean_mask_root": str(ANN["clean_masks"]),
              "propagated_mask_root": str(ANN["propagated_masks"])}


[ANN] root /kaggle/input/datasets/shanmuk4622/tire-dataset-prepared/Tire Dataset Prepared/annotations  (file says version 'v1' -- not trusted, measuring)
[ANN] as supplied: correct 17.1  worst control 18.0  margin -0.8  -> MISALIGNED
[ANN] rebuilding from the 418 hand-drawn masks + the recorded transform traces (both are in every version of the dataset)
    1000/4180
    2000/4180
    3000/4180
    4000/4180
[ANN] rebuilt 4180 propagated mask(s) in 40s
[ANN] rebuilt:     correct 35.3  worst control 21.4  margin +13.9  -> OK
[ANN] using rebuilt masks at /kaggle/temp/tyre_study/annotations/propagated/masks


## 4 — Build and validate the one-factor arms

In [6]:
# Base = CORAL, session-balanced, native resolution, full frame,
# ImageNet/SSL initialisation, full fine-tune, raw colour, wd=.05, lr=3e-4.
FACTOR_ARMS = {
  "roi_tyre": dict(roi_mode="tyre_crop", **MASK_ROOTS),
  "head_ce": dict(head_type="ce", loss_name="cross_entropy"),
  "sampler_uniform": dict(sampler_name="uniform"),
  "sampler_classweighted": dict(sampler_name="class_weighted"),
  "res224": dict(input_resolution=224),
  "res512": dict(input_resolution=512, batch_size=16),
  "transfer_random": dict(pretrained=False),
  "ft_frozen": dict(finetune_depth="frozen"),
  "wd_low": dict(weight_decay=0.01),
  "prep_clahe": dict(preprocessing="clahe"),
  "prep_gray": dict(preprocessing="grayscale"),
  "lr_low": dict(lr_initial=1e-4),
}

cfgs = []
for factor, overrides in FACTOR_ARMS.items():
    for arch in TOP3:
        if factor == "res224" and int(tl.ZOO[arch]["res"]) == 224:
            print(f"STRUCTURAL SKIP: {arch} already uses 224; no no-op res224 arm")
            continue
        if factor == "res512" and arch in tl.FIXED_224:
            print(f"STRUCTURAL SKIP: {arch} is fixed-window 224; no res512 arm")
            continue
        arm = sess.configs([arch], (1,), (1, 2, 3),
                           technique=factor, **overrides)
        for cfg in arm: tl.validate_config(cfg)
        cfgs.extend(arm)

roi_cfgs = [x for x in cfgs if x["technique"] == "roi_tyre"]
other_cfgs = [x for x in cfgs if x["technique"] != "roi_tyre"]
run_ids = [x["run_id"] for x in cfgs]
est = tl.estimate_phase(run_ids, num_workers=NUM_WORKERS)
print(f"{len(run_ids)} runs; ~{est['total_gpu_hours']:.0f} GPU-hours total")
print(tl.shard_report(run_ids, max(1, NUM_WORKERS)).to_string(index=False))

# Hugging Face is the authority. Print one compact live audit before any
# worker starts, then run_all refreshes it again at execution time.
sess.inventory.refresh(run_ids, verbose=False)
hf_done = [r for r in run_ids if sess.inventory.state(r) == "completed"]
hf_resume = [r for r in run_ids if sess.inventory.state(r) == "resumable"]
hf_absent = [r for r in run_ids if sess.inventory.state(r) == "absent"]
hf_failed = [r for r in run_ids
             if sess.inventory.status.get(r, {}).get("status") == "failed"]
print("\nPUBLIC HF Stage-B progress:",
      f"completed={len(hf_done)} resumable={len(hf_resume)} "
      f"absent={len(hf_absent)} prior_failed={len(hf_failed)}")
for rid in hf_done:
    print("  COMPLETE", rid)
for rid in hf_resume:
    print("  RESUME  ", rid, "from epoch", sess.inventory.epoch(rid))
for rid in hf_failed:
    st = sess.inventory.status[rid]
    print("  RETRY   ", rid, "after", st.get("error_type", "recorded failure"),
          "at epoch", sess.inventory.epoch(rid))
assert len(hf_done) + len(hf_resume) + len(hf_absent) == len(run_ids)


108 runs; ~53 GPU-hours total
 worker  runs  est_hours
      0    27       13.2
      1    27       13.2
      2    27       13.2
      3    27       13.2


STATUS.json: 0.00B [00:00, ?B/s]

STATUS.json: 0.00B [00:00, ?B/s]

STATUS.json: 0.00B [00:00, ?B/s]

STATUS.json: 0.00B [00:00, ?B/s]

...

STATUS.json: 0.00B [00:00, ?B/s]

STATUS.json: 0.00B [00:00, ?B/s]

STATUS.json: 0.00B [00:00, ?B/s]

STATUS.json: 0.00B [00:00, ?B/s]


PUBLIC HF Stage-B progress: completed=102 resumable=2 absent=4 prior_failed=0
  COMPLETE b-regnety016-roi_tyre-f1-s1
  COMPLETE b-regnety016-roi_tyre-f1-s2
  COMPLETE b-regnety016-roi_tyre-f1-s3
  COMPLETE b-densenet121-roi_tyre-f1-s1
  COMPLETE b-densenet121-roi_tyre-f1-s2
  COMPLETE b-densenet121-roi_tyre-f1-s3
  COMPLETE b-resnet50-roi_tyre-f1-s1
  COMPLETE b-resnet50-roi_tyre-f1-s2
  COMPLETE b-resnet50-roi_tyre-f1-s3
  COMPLETE b-regnety016-head_ce-f1-s1
  COMPLETE b-regnety016-head_ce-f1-s2
  COMPLETE b-regnety016-head_ce-f1-s3
  COMPLETE b-densenet121-head_ce-f1-s1
  COMPLETE b-densenet121-head_ce-f1-s2
  COMPLETE b-densenet121-head_ce-f1-s3
  COMPLETE b-resnet50-head_ce-f1-s1
  COMPLETE b-resnet50-head_ce-f1-s2
  COMPLETE b-resnet50-head_ce-f1-s3
  COMPLETE b-regnety016-sampler_uniform-f1-s1
  COMPLETE b-regnety016-sampler_uniform-f1-s2
  COMPLETE b-regnety016-sampler_uniform-f1-s3
  COMPLETE b-densenet121-sampler_uniform-f1-s1
  COMPLETE b-densenet121-sampler_uniform-f1-s2
  

## 5 — Run the shortcut-removing ROI control first

In [7]:
# steal_stale=False keeps FRESH work with its static owner (that is what
# stopped four accounts grabbing the same run at a simultaneous start).
# takeover_when_idle=True lets a worker that has finished its own shard pick up
# what is left, one run at a time, through a two-phase claim -- so no GPU sits
# parked while another account still has twenty runs. See docs/05 Bug 24.
roi_summaries = sess.run_all(roi_cfgs, title='Stage B — ROI control first',
                             steal_stale=False, takeover_when_idle=True,
                             isolate_runs=True)

[INV] repository holds 271 run(s); of the 9 in this notebook: 9 finished, 0 resumable


acc2_w1_387dcf.jsonl: 0.00B [00:00, ?B/s]

acct1_w0_01e0de.jsonl: 0.00B [00:00, ?B/s]

acct1_w0_051f90.jsonl: 0.00B [00:00, ?B/s]

acct1_w0_053e1b.jsonl: 0.00B [00:00, ?B/s]

...

acct4_w3_ebbbea.jsonl: 0.00B [00:00, ?B/s]

acct4_w3_f815ac.jsonl: 0.00B [00:00, ?B/s]

acct4_w3_faff64.jsonl: 0.00B [00:00, ?B/s]

acct4_w3_fd95fb.jsonl: 0.00B [00:00, ?B/s]


=== Stage B — ROI control first ===
  total in this notebook : 9
  already finished       : 9   (skipped)
  resuming mid-run       : 0
  starting from scratch  : 0
  est. GPU time for me   : ~0.0 h (credits partly-done runs)
  -> will run 0 run(s) this session



## 6 — Run the remaining one-factor arms

In [8]:
summaries = sess.run_all(other_cfgs, title='Stage B — remaining OFAT arms',
                         steal_stale=False, takeover_when_idle=True,
                         isolate_runs=True)

[INV] repository holds 271 run(s); of the 95 in this notebook: 93 finished, 2 resumable

=== Stage B — remaining OFAT arms ===
  total in this notebook : 99
  already finished       : 93   (skipped)
  resuming mid-run       : 2
  starting from scratch  : 4
  available if I go idle : 6   (claimed one at a time, only after my own 0)
  est. GPU time for me   : ~2.3 h (credits partly-done runs)
  -> will run 6 run(s) this session


--------------------------------------------------------------------------
[IDLE] my own 0 run(s) are done or running elsewhere. Taking work from the shared pool so this GPU is not parked while other accounts still have runs left.
--------------------------------------------------------------------------
[HF] flush (claim b-resnet50-prep_gray-f1-s2): 1 file(s)
[HF] commit #2: 1 file(s), 0.0 MB, 1.6s  [2/25 this hr]
[IDLE] b-resnet50-prep_gray-f1-s2: claimed after settling

[RUN] 1/6  b-resnet50-prep_gray-f1-s2   (resume from epoch 29 (was paused))
[ISOLATE] b-re

ep  29/60:   1%|          | 1/99 [00:08<14:32,  8.90s/b, acc=1.000, loss=0.0000, lr=1.88e-04]

[LIVE] b-resnet50-prep_gray-f1-s2: epoch 29/60 batch 1/99 completed in 9s -- training is active


  ep  29/60  loss 0.0001  val_acc 0.844  val_F1 0.620  val_QWK 0.9119  | 3.2m  dl 52%


ep  30/60:   0%|          | 0/99 [00:00<?, ?b/s]

...

  ep  60/60  loss 0.0001  val_acc 0.383  val_F1 0.357  val_QWK 0.5677  | 2.7m  dl 50%
[HF] flush (run completed: b-resnet50-transfer_random-f1-s3): 12 file(s)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...ample/predictions.parquet: 100%|██████████| 8.84kB / 8.84kB            


  ...try/system_samples.csv.gz:  62%|██████▏   |  274kB /  441kB            



  ...try/energy_samples.csv.gz:  88%|████████▊ | 2.77MB / 3.13MB            




  .../checkpoints/ckpt_best.pt:  45%|████▌     |  128MB /  283MB            

  ...ample/predictions.parquet: 100%|██████████| 8.84kB / 8.84kB            


  ...try/system_samples.csv.gz:  62%|██████▏   |  274kB /  441kB            



  ...try/energy_samples.csv.gz:  88%|████████▊ | 2.77MB / 3.13MB            




Processing Files (1 / 4)      :  23%|██▎       |  131MB /  569MB,   ???B/s  

  ...ample/predictions.parquet: 100%|██████████| 8.84kB / 8.84kB            


  ...try/system_samples.csv.gz:  62%|██████▏   |  274kB /  441kB            



  ...try/energy_samples.csv.gz:  88%|████████▊ | 2.77MB / 

[HF] commit #11: 12 file(s), 570.0 MB, 6.5s  [5/25 this hr]
[TRAIN] b-resnet50-transfer_random-f1-s3  ->  completed  best QWK 0.9119  (2.76h)
[SESSION] final flush -- blocking until HuggingFace confirms
[SESSION] done. commits=11 failures=0 pushed=6235 MB
[LIFE] flush triggered by atexit
[FLUSH] emergency flush (atexit)


STATUS.json: 0.00B [00:00, ?B/s]

[ISOLATE] b-resnet50-transfer_random-f1-s3: child exited rc=0; status=completed epoch=60. Its process memory is now fully reclaimed.
[DISK] b-resnet50-transfer_random-f1-s3: freed 0.57 GB locally (both checkpoints confirmed on HuggingFace)

                          run_id     arch  fold  seed    status  best_val_qwk  best_val_f1_macro  best_val_acc  epochs_trained  total_wall_seconds  total_energy_wh
      b-resnet50-prep_gray-f1-s2 resnet50     1     2 completed      0.915612           0.607143       0.84375              60        10269.445467       288.201316
b-resnet50-transfer_random-f1-s3 resnet50     1     3 completed      0.911894           0.620370       0.84375              60         9924.382453       284.390223


## 7 — Effects relative to the matching Stage-A base runs

In [9]:
import numpy as np, pandas as pd
base_ids = [f"a-{a}-base-f1-s{s}" for a in TOP3 for s in (1,2,3)]
A = sess.aggregate_remote(base_ids, verbose=False)
B = sess.aggregate_remote(run_ids, verbose=False)
if len(A) and len(B):
    base = A.set_index(["arch", "fold", "seed"])["best_val_f1_macro"]
    rows = []
    for r in B[B.status == "completed"].itertuples():
        key = (r.arch, int(r.fold), int(r.seed))
        if key in base.index:
            rows.append({"arch": r.arch, "factor": r.technique, "fold": r.fold,
                         "seed": r.seed, "f1": r.best_val_f1_macro,
                         "delta_vs_stage_a": r.best_val_f1_macro - float(base.loc[key])})
    E = pd.DataFrame(rows)
    print(E.groupby(["arch", "factor"]).agg(
        n=("delta_vs_stage_a", "size"), mean_delta=("delta_vs_stage_a", "mean"),
        sd=("delta_vs_stage_a", "std")).round(4).to_string())
    out = Path(sess.stage_dir) / "tables" / "stage_b_effects.csv"
    E.to_csv(out, index=False); sess.uploader.enqueue(out, "tables/stage_b_effects.csv", force=True)
    sess.push_now("Stage B effects updated")
else:
    print("No completed Stage-B arms yet; rerun this cell after progress accumulates.")


final.csv:   0%|          | 0.00/695 [00:00<?, ?B/s]

final.csv:   0%|          | 0.00/640 [00:00<?, ?B/s]

final.csv:   0%|          | 0.00/700 [00:00<?, ?B/s]

final.csv:   0%|          | 0.00/696 [00:00<?, ?B/s]

final.csv:   0%|          | 0.00/656 [00:00<?, ?B/s]

...

final.csv: 0.00B [00:00, ?B/s]

final.csv: 0.00B [00:00, ?B/s]

final.csv: 0.00B [00:00, ?B/s]

final.csv: 0.00B [00:00, ?B/s]

                                   n  mean_delta      sd
arch        factor                                      
densenet121 ft_frozen              3     -0.0943  0.1978
            head_ce                3      0.0817  0.2112
            lr_low                 3     -0.0717  0.0687
            prep_clahe             3     -0.0713  0.0692
            prep_gray              3     -0.0694  0.0719
            res224                 3      0.0414  0.3246
            res512                 3     -0.0694  0.0719
            roi_tyre               3     -0.0700  0.0712
            sampler_classweighted  3      0.0196  0.2249
            sampler_uniform        3      0.0115  0.1688
            transfer_random        3     -0.0034  0.2031
            wd_low                 3     -0.0523  0.0346
regnety016  ft_frozen              3     -0.1767  0.2177
            head_ce                3     -0.1860  0.1692
            lr_low                 3     -0.1845  0.1691
            prep_clahe         

## 8 — Final public verification

In [10]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

# Draining the upload queue is NOT the same as the files being on HuggingFace.
# Ask the repository before you close this tab.
#
# Four states. FINISHED and RESUMABLE are both safe -- a run paused
# at epoch 34 whose ckpt_last.pt is on HF loses nothing when you close the tab.
# NOT STARTED means no run files exist and no work was lost. Only AT RISK means
# partial artifacts exist without a usable checkpoint and needs attention.
sess.confirm_on_hf(run_ids)


[SESSION] final flush -- blocking until HuggingFace confirms
[SESSION] done. commits=5 failures=0 pushed=0 MB


STATUS.json: 0.00B [00:00, ?B/s]

STATUS.json: 0.00B [00:00, ?B/s]

                                   run_id    on_hf  epoch  missing_files
              b-regnety016-roi_tyre-f1-s1 FINISHED     60              0
              b-regnety016-roi_tyre-f1-s2 FINISHED     60              0
              b-regnety016-roi_tyre-f1-s3 FINISHED     60              0
             b-densenet121-roi_tyre-f1-s1 FINISHED     60              0
             b-densenet121-roi_tyre-f1-s2 FINISHED     60              0
             b-densenet121-roi_tyre-f1-s3 FINISHED     60              0
                b-resnet50-roi_tyre-f1-s1 FINISHED     60              0
                b-resnet50-roi_tyre-f1-s2 FINISHED     60              0
                b-resnet50-roi_tyre-f1-s3 FINISHED     60              0
               b-regnety016-head_ce-f1-s1 FINISHED     60              0
               b-regnety016-head_ce-f1-s2 FINISHED     60              0
               b-regnety016-head_ce-f1-s3 FINISHED     60              0
              b-densenet121-head_ce-f1-s1 FINISHED 

,run_id,on_hf,epoch,missing_files
0,b-regnety016-roi_tyre-f1-s1,FINISHED,60,0
1,b-regnety016-roi_tyre-f1-s2,FINISHED,60,0
2,b-regnety016-roi_tyre-f1-s3,FINISHED,60,0
3,b-densenet121-roi_tyre-f1-s1,FINISHED,60,0
4,b-densenet121-roi_tyre-f1-s2,FINISHED,60,0
...,...,...,...,...
103,b-densenet121-lr_low-f1-s2,FINISHED,60,0
104,b-densenet121-lr_low-f1-s3,FINISHED,60,0
105,b-resnet50-lr_low-f1-s1,FINISHED,60,0
106,b-resnet50-lr_low-f1-s2,FINISHED,60,0
